# 02 — Dataset Validation (Versioned)

Bu notebook, `01_dataset_creation_versioned.ipynb` tarafından üretilen
**immutable raw dataset snapshot'larından belirli bir sürümü** doğrulamak için kullanılır.

## Temel prensip

Bu notebook `questions_master.json` okumaz.

Önce açıkça bir raw dataset sürümü seçilir:

```python
RAW_DATASET_VERSION = "raw_v002"
```

Ardından yalnızca:

```text
data/raw/snapshots/raw_v002.json
```

üzerinde çalışılır.

Böylece veri üretimi başka bir yerde devam ederken validation girdisi değişmez.

## Çıktılar

Örneğin `raw_v002` seçilirse:

```text
data/validation_work/raw_v002_validation.xlsx
data/validated/validated_v002.json
data/validated/validated_v002_manifest.json
```

oluşturulur.

## Artımlı validation

Daha sonra `raw_v003`, `raw_v004` gibi daha büyük sürümlere geçtiğinizde,
önceki sürümlerde doğrulanmış ve **içeriği değişmemiş** soruların manuel validation
alanları otomatik taşınır. Böylece aynı soruları tekrar tekrar doğrulamak gerekmez.

## 1. Google Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Proje yolları ve raw sürüm seçimi

In [ ]:
from pathlib import Path
import re

PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

RAW_SNAPSHOTS_DIR = PROJECT_ROOT / "data" / "raw" / "snapshots"
RAW_MANIFESTS_DIR = RAW_SNAPSHOTS_DIR / "manifests"
VALIDATION_WORKDIR = PROJECT_ROOT / "data" / "validation_work"
DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"

for directory in [VALIDATION_WORKDIR, DATA_VALIDATED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ============================================================
# ANA SEÇİM
# ============================================================
RAW_DATASET_VERSION = "raw_v002"
# ============================================================

if not re.fullmatch(r"raw_v\d{3}", RAW_DATASET_VERSION):
    raise ValueError("RAW_DATASET_VERSION 'raw_v002' biçiminde olmalıdır.")

VALIDATED_DATASET_VERSION = RAW_DATASET_VERSION.replace("raw_", "validated_")

RAW_JSON_PATH = RAW_SNAPSHOTS_DIR / f"{RAW_DATASET_VERSION}.json"
RAW_MANIFEST_PATH = RAW_MANIFESTS_DIR / f"{RAW_DATASET_VERSION}_manifest.json"

VALIDATION_XLSX_PATH = (
    VALIDATION_WORKDIR / f"{RAW_DATASET_VERSION}_validation.xlsx"
)

VALIDATED_JSON_PATH = (
    DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}.json"
)

VALIDATED_MANIFEST_PATH = (
    DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}_manifest.json"
)

print("Raw version:", RAW_DATASET_VERSION)
print("Input:", RAW_JSON_PATH)
print("Validation workbook:", VALIDATION_XLSX_PATH)
print("Validated output:", VALIDATED_JSON_PATH)

## 3. Mevcut raw sürümleri listele

In [ ]:
import json
import pandas as pd

def list_raw_versions():
    rows = []

    for path in sorted(RAW_SNAPSHOTS_DIR.glob("raw_v*.json")):
        match = re.fullmatch(r"raw_v(\d{3})\.json", path.name)

        if not match:
            continue

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows.append({
            "raw_version": path.stem,
            "n_questions": len(data),
            "path": str(path),
            "manifest_exists": (
                RAW_MANIFESTS_DIR / f"{path.stem}_manifest.json"
            ).exists(),
        })

    return pd.DataFrame(rows)

raw_versions_df = list_raw_versions()
display(raw_versions_df)

if RAW_DATASET_VERSION not in set(raw_versions_df["raw_version"]):
    raise FileNotFoundError(
        f"Seçilen raw sürüm bulunamadı: {RAW_DATASET_VERSION}"
    )

## 4. Raw snapshot bütünlüğünü doğrula

In [ ]:
import hashlib
from collections import Counter

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

if not RAW_JSON_PATH.exists():
    raise FileNotFoundError(RAW_JSON_PATH)

if not RAW_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Raw manifest bulunamadı: {RAW_MANIFEST_PATH}"
    )

with open(RAW_JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

with open(RAW_MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_manifest = json.load(f)

if not isinstance(raw_data, list):
    raise ValueError("Raw dataset JSON root list olmalıdır.")

actual_sha256 = file_sha256(RAW_JSON_PATH)
expected_sha256 = raw_manifest.get("file_sha256")

if actual_sha256 != expected_sha256:
    raise RuntimeError(
        "KRİTİK: Raw snapshot SHA-256 manifest ile uyuşmuyor."
    )

if len(raw_data) != raw_manifest.get("n_questions"):
    raise RuntimeError(
        "Raw snapshot soru sayısı manifest ile uyuşmuyor."
    )

print("RAW SNAPSHOT BÜTÜNLÜĞÜ BAŞARILI")
print("Version:", RAW_DATASET_VERSION)
print("Soru sayısı:", len(raw_data))
print("SHA-256:", actual_sha256)

## 5. Raw dataset'i DataFrame olarak yükle

In [ ]:
df = pd.DataFrame(raw_data)

print("Validation girdisi:", RAW_DATASET_VERSION)
print("Toplam aday soru:", len(df))
display(df.head())

## 6. Beklenen şema ve sabitler

In [ ]:
EXPECTED_COLUMNS = [
    "question_id",
    "question",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
    "source_type",
    "source_name",
    "source_reference",
    "verification_status",
    "notes",
    "batch_id",
    "created_at_utc",
]

ANSWER_TYPES = {
    "ENTITY_PERSON",
    "ENTITY_LOCATION",
    "ENTITY_ORGANIZATION",
    "ENTITY_OTHER",
    "DATE",
    "NUMBER",
    "NUMBER_WITH_UNIT",
    "SHORT_PHRASE",
    "BINARY",
}

VALID_VERIFICATION_STATUSES = {
    "unverified",
    "verified",
    "rejected",
}

missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Raw dataset içinde eksik sütunlar var: {missing_columns}"
    )

print("Şema kontrolü başarılı.")

## 7. Normalize ve fingerprint yardımcıları

In [ ]:
import unicodedata

def normalize_text_for_validation(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("İ", "i").replace("I", "ı")
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def canonical_record_for_validation(row):
    aliases = row.get("accepted_aliases", [])

    if not isinstance(aliases, list):
        aliases = []

    payload = {
        "question": row.get("question", ""),
        "reference_answer": row.get("reference_answer", ""),
        "accepted_aliases": aliases,
        "answer_type": row.get("answer_type", ""),
        "category": row.get("category", ""),
    }

    return json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def record_validation_fingerprint(row):
    payload = canonical_record_for_validation(row)

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()

## 8. Yapısal bütünlük kontrolleri

In [ ]:
structural_issues = []

if df["question_id"].isna().any():
    structural_issues.append("Eksik question_id var.")

if df["question_id"].duplicated().any():
    structural_issues.append("Duplicate question_id var.")

for col in ["question", "reference_answer", "answer_type", "category"]:
    bad = df[
        df[col].isna()
        |
        (df[col].astype(str).str.strip() == "")
    ]

    if not bad.empty:
        structural_issues.append(
            f"{col}: {len(bad)} boş/geçersiz kayıt"
        )

invalid_answer_types = df[
    ~df["answer_type"].isin(ANSWER_TYPES)
]

if not invalid_answer_types.empty:
    structural_issues.append(
        f"Geçersiz answer_type: {len(invalid_answer_types)} kayıt"
    )

invalid_statuses = df[
    ~df["verification_status"].isin(VALID_VERIFICATION_STATUSES)
]

if not invalid_statuses.empty:
    structural_issues.append(
        f"Geçersiz verification_status: {len(invalid_statuses)} kayıt"
    )

def alias_list_problem(value):
    if not isinstance(value, list) or len(value) == 0:
        return True

    return any(
        not isinstance(alias, str) or not alias.strip()
        for alias in value
    )

alias_problem_mask = df["accepted_aliases"].apply(alias_list_problem)

if alias_problem_mask.any():
    structural_issues.append(
        f"Geçersiz accepted_aliases: {int(alias_problem_mask.sum())} kayıt"
    )

if structural_issues:
    print("YAPISAL PROBLEMLER BULUNDU")
    for issue in structural_issues:
        print("-", issue)
else:
    print("YAPISAL BÜTÜNLÜK KONTROLÜ BAŞARILI")

## 9. Exact duplicate kontrolü

In [ ]:
df["normalized_question"] = df[
    "question"
].apply(normalize_text_for_validation)

exact_duplicate_mask = df[
    "normalized_question"
].duplicated(keep=False)

exact_duplicates = df.loc[
    exact_duplicate_mask,
    [
        "question_id",
        "question",
        "reference_answer",
        "answer_type",
        "category",
        "normalized_question",
    ],
].sort_values("normalized_question")

print("Exact duplicate soru kaydı:", len(exact_duplicates))
display(exact_duplicates)

## 10. Near-duplicate adaylarını bul

In [ ]:
!pip -q install rapidfuzz

In [ ]:
from rapidfuzz.fuzz import ratio

NEAR_DUPLICATE_THRESHOLD = 88
near_duplicate_pairs = []

records = df[
    ["question_id", "question", "normalized_question"]
].to_dict("records")

for i in range(len(records)):
    for j in range(i + 1, len(records)):
        q1 = records[i]["normalized_question"]
        q2 = records[j]["normalized_question"]

        if not q1 or not q2 or q1 == q2:
            continue

        score = ratio(q1, q2)

        if score >= NEAR_DUPLICATE_THRESHOLD:
            near_duplicate_pairs.append({
                "question_id_1": records[i]["question_id"],
                "question_1": records[i]["question"],
                "question_id_2": records[j]["question_id"],
                "question_2": records[j]["question"],
                "similarity": score,
            })

near_duplicates_df = pd.DataFrame(near_duplicate_pairs)

if not near_duplicates_df.empty:
    near_duplicates_df = near_duplicates_df.sort_values(
        "similarity",
        ascending=False,
    ).reset_index(drop=True)

print("Near-duplicate aday çifti:", len(near_duplicates_df))
display(near_duplicates_df.head(100))

## 11. Reference answer / alias tutarlılığı

In [ ]:
alias_issues = []

for _, row in df.iterrows():
    aliases = row["accepted_aliases"]

    if not isinstance(aliases, list):
        continue

    normalized_aliases = [
        normalize_text_for_validation(alias)
        for alias in aliases
        if isinstance(alias, str)
    ]

    reference_norm = normalize_text_for_validation(
        row["reference_answer"]
    )

    if reference_norm not in normalized_aliases:
        alias_issues.append({
            "question_id": row["question_id"],
            "issue": "reference_answer alias listesinde yok",
            "reference_answer": row["reference_answer"],
            "accepted_aliases": aliases,
        })

    if len(normalized_aliases) != len(set(normalized_aliases)):
        alias_issues.append({
            "question_id": row["question_id"],
            "issue": "duplicate alias",
            "reference_answer": row["reference_answer"],
            "accepted_aliases": aliases,
        })

alias_issues_df = pd.DataFrame(alias_issues)

print("Alias problemi:", len(alias_issues_df))
display(alias_issues_df)

## 12. Temporal / ambiguity risk heuristikleri

In [ ]:
TEMPORAL_PATTERNS = [
    r"günümüzde",
    r"şu anda",
    r"şu an",
    r"halen",
    r"hâlen",
    r"mevcut",
    r"güncel",
    r"en son",
    r"son olarak",
    r"bugün",
    r"bu yıl",
    r"en zengin",
    r"başkanı kim",
    r"ceo",
]

AMBIGUITY_PATTERNS = [
    r"en iyi",
    r"en ünlü",
    r"en önemli",
    r"en başarılı",
    r"genellikle",
    r"çoğunlukla",
]

def matches_any_pattern(text, patterns):
    normalized = normalize_text_for_validation(text)

    return any(
        re.search(pattern, normalized)
        for pattern in patterns
    )

df["temporal_risk_flag"] = df["question"].apply(
    lambda text: int(
        matches_any_pattern(text, TEMPORAL_PATTERNS)
    )
)

df["ambiguity_risk_flag"] = df["question"].apply(
    lambda text: int(
        matches_any_pattern(text, AMBIGUITY_PATTERNS)
    )
)

risk_df = df[
    (df["temporal_risk_flag"] == 1)
    |
    (df["ambiguity_risk_flag"] == 1)
][
    [
        "question_id",
        "question",
        "reference_answer",
        "temporal_risk_flag",
        "ambiguity_risk_flag",
    ]
]

print("Risk bayrağı alan kayıt:", len(risk_df))
display(risk_df)

## 13. Veri dağılımları

In [ ]:
def distribution_table(column):
    counts = (
        df[column]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

    counts["ratio"] = counts["count"] / len(df)
    return counts

print("ANSWER TYPE")
display(distribution_table("answer_type"))

print("\nCATEGORY")
display(distribution_table("category"))

print("\nVERIFICATION STATUS")
display(distribution_table("verification_status"))

print("\nSOURCE TYPE")
display(distribution_table("source_type"))

# 14. Önceki validation sonuçlarını yeniden kullan

Yeni raw sürüm çoğunlukla eski sorular + yeni batch şeklinde büyüyecektir.

Örneğin:

```text
raw_v002 = 120 soru
raw_v003 = 180 soru
```

Aynı `question_id` için soru, reference answer, alias listesi, answer type ve kategori
değişmemişse eski validation kararı yeniden kullanılabilir.

Bunu güvenli yapmak için `record_fingerprint` kullanılır.

In [ ]:
MANUAL_COLUMNS = [
    "factually_correct",
    "single_unambiguous_answer",
    "time_stable",
    "aliases_valid",
    "verification_source",
    "validation_notes",
    "validation_decision",
]

df["record_fingerprint"] = df.apply(
    lambda row: record_validation_fingerprint(row.to_dict()),
    axis=1,
)

def raw_version_number(version_name):
    match = re.fullmatch(r"raw_v(\d{3})", version_name)

    if not match:
        raise ValueError(version_name)

    return int(match.group(1))

def previous_validation_workbooks(current_version):
    current_number = raw_version_number(current_version)
    candidates = []

    for path in VALIDATION_WORKDIR.glob("raw_v*_validation.xlsx"):
        match = re.fullmatch(
            r"raw_v(\d{3})_validation\.xlsx",
            path.name,
        )

        if not match:
            continue

        version_number = int(match.group(1))

        if version_number < current_number:
            candidates.append((version_number, path))

    return sorted(candidates, reverse=True)

def load_reusable_manual_validations(current_version):
    reusable = {}

    for version_number, path in previous_validation_workbooks(current_version):
        try:
            old_df = pd.read_excel(path, sheet_name=0)
        except Exception as exc:
            print(
                "UYARI: önceki validation workbook okunamadı:",
                path,
                repr(exc),
            )
            continue

        required = {
            "question_id",
            "record_fingerprint",
            *MANUAL_COLUMNS,
        }

        if not required.issubset(set(old_df.columns)):
            continue

        for _, row in old_df.iterrows():
            qid = str(row["question_id"]).strip()

            if qid in reusable:
                continue

            decision = str(
                row.get("validation_decision", "")
            ).strip().upper()

            if decision not in {"ACCEPT", "REJECT", "REVISE"}:
                continue

            record = {
                "record_fingerprint": str(
                    row["record_fingerprint"]
                ).strip(),
                "_source_validation_version": (
                    f"raw_v{version_number:03d}"
                ),
            }

            for col in MANUAL_COLUMNS:
                record[col] = row.get(col, "")

            reusable[qid] = record

    return reusable

reusable_validations = load_reusable_manual_validations(
    RAW_DATASET_VERSION
)

print(
    "Önceki validation'lardan yeniden kullanılabilir kayıt:",
    len(reusable_validations),
)

## 15. Version-specific validation workbook oluştur

Eğer bu raw sürüme ait workbook zaten varsa **üzerine yazılmaz**.
Bu, Numbers/Excel ile yaptığınız manuel çalışmanın silinmesini engeller.

İlk oluşturulurken önceki validation'lardan fingerprint eşleşen kayıtlar taşınır.

In [ ]:
def build_validation_workbook():
    if VALIDATION_XLSX_PATH.exists():
        print("Validation workbook zaten mevcut.")
        print("Üzerine yazılmadı:", VALIDATION_XLSX_PATH)

        return pd.read_excel(
            VALIDATION_XLSX_PATH,
            sheet_name=0,
        )

    work_df = df.copy()

    work_df["accepted_aliases"] = work_df[
        "accepted_aliases"
    ].apply(
        lambda aliases:
        " | ".join(aliases)
        if isinstance(aliases, list)
        else ""
    )

    for col in MANUAL_COLUMNS:
        work_df[col] = ""

    work_df["carried_forward_from"] = ""

    carried_count = 0

    for idx, row in work_df.iterrows():
        qid = str(row["question_id"]).strip()
        old = reusable_validations.get(qid)

        if not old:
            continue

        if (
            str(row["record_fingerprint"]).strip()
            != str(old["record_fingerprint"]).strip()
        ):
            continue

        for col in MANUAL_COLUMNS:
            work_df.at[idx, col] = old.get(col, "")

        work_df.at[
            idx,
            "carried_forward_from",
        ] = old["_source_validation_version"]

        carried_count += 1

    workbook_columns = [
        "question_id",
        "record_fingerprint",
        "question",
        "reference_answer",
        "accepted_aliases",
        "answer_type",
        "category",
        "source_type",
        "source_name",
        "source_reference",
        "verification_status",
        "temporal_risk_flag",
        "ambiguity_risk_flag",
        "carried_forward_from",
        *MANUAL_COLUMNS,
    ]

    work_df = work_df[workbook_columns]

    work_df.to_excel(
        VALIDATION_XLSX_PATH,
        index=False,
    )

    print("Yeni validation workbook oluşturuldu:")
    print(VALIDATION_XLSX_PATH)
    print("Önceki sürümden taşınan:", carried_count)
    print(
        "Yeni/yeniden doğrulanması gereken:",
        len(work_df) - carried_count,
    )

    return work_df

validation_work_df = build_validation_workbook()
display(validation_work_df.head())

## 16. Manuel validation nasıl doldurulur?

Workbook'u Numbers veya Excel ile açın.

- `factually_correct`: `1/0`
- `single_unambiguous_answer`: `1/0`
- `time_stable`: `1/0`
- `aliases_valid`: `1/0`
- `verification_source`: güvenilir kaynak / URL / bibliyografik referans
- `validation_notes`: gerekiyorsa açıklama
- `validation_decision`: `ACCEPT`, `REVISE`, `REJECT`

`ACCEPT` için dört kalite alanının da `1` olması ve `verification_source`
alanının dolu olması gerekir.

Workbook'u aynı `.xlsx` dosyası olarak kaydedin.

## 17. Manuel validation workbook'unu tekrar yükle

In [ ]:
if not VALIDATION_XLSX_PATH.exists():
    raise FileNotFoundError(VALIDATION_XLSX_PATH)

review_df = pd.read_excel(
    VALIDATION_XLSX_PATH,
    sheet_name=0,
)

print("Yüklenen validation kayıt sayısı:", len(review_df))
display(review_df.head())

## 18. Workbook'un seçili raw sürümle bütünlüğünü doğrula

In [ ]:
expected_fingerprints = {
    row["question_id"]: row["record_fingerprint"]
    for _, row in df.iterrows()
}

workbook_integrity_problems = []

if len(review_df) != len(df):
    workbook_integrity_problems.append(
        "Workbook satır sayısı raw dataset ile uyuşmuyor."
    )

if review_df["question_id"].duplicated().any():
    workbook_integrity_problems.append(
        "Workbook içinde duplicate question_id var."
    )

for _, row in review_df.iterrows():
    qid = str(row.get("question_id", "")).strip()
    fp = str(row.get("record_fingerprint", "")).strip()

    if qid not in expected_fingerprints:
        workbook_integrity_problems.append(
            f"Raw dataset'te olmayan ID: {qid}"
        )
        continue

    if fp != expected_fingerprints[qid]:
        workbook_integrity_problems.append(
            f"Fingerprint uyuşmazlığı: {qid}"
        )

if workbook_integrity_problems:
    print("WORKBOOK BÜTÜNLÜK KONTROLÜ BAŞARISIZ")
    for problem in workbook_integrity_problems:
        print("-", problem)
else:
    print("WORKBOOK BÜTÜNLÜK KONTROLÜ BAŞARILI")

## 19. Manuel validation kararlarını doğrula

In [ ]:
VALID_DECISIONS = {"ACCEPT", "REVISE", "REJECT"}

review_df["validation_decision"] = (
    review_df["validation_decision"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

review_problems = []

if workbook_integrity_problems:
    review_problems.extend(workbook_integrity_problems)

invalid_decisions = review_df[
    ~review_df["validation_decision"].isin(VALID_DECISIONS)
]

if not invalid_decisions.empty:
    review_problems.append(
        f"Kararı eksik/geçersiz kayıt: {len(invalid_decisions)}"
    )

QUALITY_COLUMNS = [
    "factually_correct",
    "single_unambiguous_answer",
    "time_stable",
    "aliases_valid",
]

for col in QUALITY_COLUMNS:
    review_df[col] = pd.to_numeric(
        review_df[col],
        errors="coerce",
    )

accepted_df = review_df[
    review_df["validation_decision"] == "ACCEPT"
].copy()

for col in QUALITY_COLUMNS:
    bad = accepted_df[accepted_df[col] != 1]

    if not bad.empty:
        review_problems.append(
            f"ACCEPT kayıtlarında {col} != 1: {len(bad)}"
        )

empty_source = accepted_df[
    accepted_df["verification_source"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
]

if not empty_source.empty:
    review_problems.append(
        f"ACCEPT kayıtlarında verification_source boş: "
        f"{len(empty_source)}"
    )

print("=" * 70)

if review_problems:
    print("MANUEL VALIDATION HENÜZ TAMAMLANMADI")

    for problem in review_problems:
        print("-", problem)

    print()
    print(
        "Workbook'u doldurup kaydettikten sonra "
        "17-19. hücreleri tekrar çalıştırın."
    )
else:
    print("MANUEL VALIDATION KONTROLÜ BAŞARILI")
    print(
        "ACCEPT:",
        (review_df["validation_decision"] == "ACCEPT").sum(),
    )
    print(
        "REVISE:",
        (review_df["validation_decision"] == "REVISE").sum(),
    )
    print(
        "REJECT:",
        (review_df["validation_decision"] == "REJECT").sum(),
    )

## 20. REVISE / REJECT kayıtlarını incele

In [ ]:
needs_revision = review_df[
    review_df["validation_decision"].isin(
        ["REVISE", "REJECT"]
    )
][
    [
        "question_id",
        "question",
        "reference_answer",
        "validation_decision",
        "validation_notes",
    ]
]

display(needs_revision)

# 21. Validated dataset oluştur

Yalnızca `ACCEPT` kayıtları alınır.

Örneğin:

```text
raw_v002
    ↓
validated_v002
```

`validated_v002.json` zaten varsa üzerine yazılmaz.

In [ ]:
from datetime import datetime, timezone

if review_problems:
    raise ValueError(
        "Validation tamamlanmadan validated dataset üretilemez."
    )

if VALIDATED_JSON_PATH.exists():
    raise FileExistsError(
        f"Validated dataset zaten mevcut ve immutable kabul ediliyor: "
        f"{VALIDATED_JSON_PATH}"
    )

raw_lookup = {
    item["question_id"]: item
    for item in raw_data
}

final_records = []

for _, review_row in accepted_df.iterrows():
    question_id = str(
        review_row["question_id"]
    ).strip()

    if question_id not in raw_lookup:
        raise KeyError(
            f"Raw dataset içinde bulunamadı: {question_id}"
        )

    record = dict(raw_lookup[question_id])
    record["verification_status"] = "verified"

    record["validation"] = {
        "source_raw_version": RAW_DATASET_VERSION,
        "record_fingerprint": str(
            review_row["record_fingerprint"]
        ).strip(),
        "factually_correct": 1,
        "single_unambiguous_answer": 1,
        "time_stable": 1,
        "aliases_valid": 1,
        "verification_source": str(
            review_row["verification_source"]
        ).strip(),
        "validation_notes": (
            ""
            if pd.isna(review_row["validation_notes"])
            else str(review_row["validation_notes"]).strip()
        ),
        "carried_forward_from": (
            ""
            if pd.isna(review_row.get("carried_forward_from", ""))
            else str(
                review_row.get("carried_forward_from", "")
            ).strip()
        ),
        "validated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    final_records.append(record)

with open(
    VALIDATED_JSON_PATH,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        final_records,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Validated dataset oluşturuldu:")
print(VALIDATED_JSON_PATH)
print("ACCEPT edilen soru:", len(final_records))

## 22. Validated dataset final bütünlük kontrolü

In [ ]:
with open(
    VALIDATED_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    final_data = json.load(f)

final_df = pd.DataFrame(final_data)
final_problems = []

if final_df.empty:
    final_problems.append("Validated dataset boş.")

if not final_df.empty and final_df["question_id"].duplicated().any():
    final_problems.append("Duplicate question_id var.")

if not final_df.empty:
    normalized_final = final_df["question"].apply(
        normalize_text_for_validation
    )

    if normalized_final.duplicated().any():
        final_problems.append("Exact duplicate soru var.")

    if not (
        final_df["verification_status"] == "verified"
    ).all():
        final_problems.append("Verified olmayan kayıt var.")

    if "validation" not in final_df.columns:
        final_problems.append("Validation metadata eksik.")

if final_problems:
    print("VALIDATED DATASET KONTROLÜ BAŞARISIZ")
    for problem in final_problems:
        print("-", problem)
else:
    print("VALIDATED DATASET KONTROLÜ BAŞARILI")
    print("Raw version:", RAW_DATASET_VERSION)
    print("Validated version:", VALIDATED_DATASET_VERSION)
    print("Raw soru:", len(raw_data))
    print("Accepted soru:", len(final_data))
    print(
        "Excluded:",
        len(raw_data) - len(final_data),
    )

## 23. Validated dataset dağılımları

In [ ]:
def validated_distribution(column):
    result = (
        final_df[column]
        .value_counts()
        .rename("count")
        .to_frame()
    )

    result["ratio"] = (
        result["count"] / len(final_df)
    )

    return result

print("ANSWER TYPE")
display(validated_distribution("answer_type"))

print("\nCATEGORY")
display(validated_distribution("category"))

## 24. Validated dataset manifest + checksum

In [ ]:
validated_sha256 = file_sha256(VALIDATED_JSON_PATH)

validated_manifest = {
    "validated_dataset_version": VALIDATED_DATASET_VERSION,
    "validated_dataset_file": VALIDATED_JSON_PATH.name,
    "source_raw_version": RAW_DATASET_VERSION,
    "source_raw_file": RAW_JSON_PATH.name,
    "source_raw_file_sha256": actual_sha256,
    "source_raw_manifest": RAW_MANIFEST_PATH.name,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_raw_questions": len(raw_data),
    "n_validated_questions": len(final_data),
    "n_excluded_questions": len(raw_data) - len(final_data),
    "validated_file_sha256": validated_sha256,
    "answer_type_distribution": dict(
        Counter(
            item["answer_type"]
            for item in final_data
        )
    ),
    "category_distribution": dict(
        Counter(
            item["category"]
            for item in final_data
        )
    ),
}

if VALIDATED_MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Validated manifest zaten mevcut: "
        f"{VALIDATED_MANIFEST_PATH}"
    )

with open(
    VALIDATED_MANIFEST_PATH,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        validated_manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Validated SHA-256:", validated_sha256)
print("Manifest:", VALIDATED_MANIFEST_PATH)

# 25. Paralel veri üretiminde kullanım

Bugün:

```text
raw_v002 → validated_v002
```

ile sonraki notebook'ları geliştirebilirsiniz.

Başka chatte yeni batch'ler eklenirken:

```text
raw_v003
raw_v004
...
```

oluşmaya devam eder.

Daha sonra örneğin `raw_v004` doğrulanacaksa yalnızca:

```python
RAW_DATASET_VERSION = "raw_v004"
```

yapılır.

Yeni workbook oluşturulurken önceki sürümdeki içeriği değişmemiş soruların
manuel validation alanları otomatik taşınır. Yalnızca yeni veya değiştirilmiş
soruları doğrulamak gerekir.

Sonraki model notebook'ları da living master yerine açık bir:

```text
validated_vXXX
```

sürümü kullanacaktır.

# Notebook 02 tamamlanma kriterleri

Belirli bir raw sürüm için:

- [ ] Raw JSON / manifest checksum doğrulandı
- [ ] Yapısal validation hatasız
- [ ] Duplicate ID yok
- [ ] Exact duplicate yok
- [ ] Near-duplicate adayları incelendi
- [ ] Alias sorunları çözüldü
- [ ] Temporal / ambiguity riskleri incelendi
- [ ] Yeni/değişmiş sorular factual olarak doğrulandı
- [ ] ACCEPT soruların dört kalite alanı `1`
- [ ] ACCEPT sorularda `verification_source` mevcut
- [ ] `validated_vXXX.json` üretildi
- [ ] Validated manifest ve checksum üretildi
- [ ] Validated dosya immutable bırakıldı